# Publish training and inference from source

Fill in **bucket, role ARN and image** below. Run the cells in order to check basic
pipeline access, upload source, and publish **both** pipelines. Then run training
and inference from Studio, or inspect them with `03G_inspect_source_runs.ipynb`.

The version-label permission check prints its original AWS error and continues.
Publication uses ordinary CreatePipeline/UpdatePipeline calls and does not need
ListPipelineVersions or UpdatePipelineVersion. Numeric versions still come from
SageMaker; the source identity remains in each pipeline definition.


### 1. Required inputs

- Enter the bucket name, with or without s3://.
- Paste the exact IAM role ARN, including its path; it is used as both the pipeline role and the job role.
- Paste the image URI; a repository address with no version on the end also works.
- The namespace names the two pipelines and their folder in the bucket; use a different namespace for CI deployment.

In [ ]:
bucket = ""                        # bucket name, with or without s3://
role_arn = ""                      # exact IAM role ARN; used as pipeline role and job role
image_uri = ""                     # repository address, repository:tag or repository@sha256:...

namespace = "source-dev"
model_ids = None                   # Discover model folders; or select ["usecase1", "usecase2"].
batch_runtime = "transform"        # Native Batch Transform; test the image with 00I first.
                                  # "processing" runs the same predictor in a Processing job.


### 2. Load this checkout

- Find the application folder from this notebook's working directory.
- Import the shared source upload and pipeline definition functions.
- Use the one role for pipelines and jobs, and `s3://<bucket>/<namespace>` as the application folder.
- Print the notebook caller and the supplied role separately; no upload or creation happens yet.


In [ ]:
from pathlib import Path
import json, sys
import boto3
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "mlops-models/src").is_dir())
sys.path[:0] = [str(ROOT / "mlops-models"), str(ROOT / "mlops-models/src")]
from mlops_models.pipelines.source import (
    source_identity, source_location, upload_source, upload_examples, build_definitions,
)
from mlops_skeleton.names import discover_models
model_ids = model_ids or [folder.name for folder in discover_models()]
session = boto3.Session()
sm, s3 = session.client("sagemaker"), session.client("s3")
print(session.client("sts").get_caller_identity())
print("Source:", ROOT / "mlops-models/src/mlops_models")
for model in model_ids:
    print(model, (ROOT / "mlops-models/src/mlops_models/models" / model / "model-config.json").is_file())
bucket = bucket.strip().removeprefix("s3://").rstrip("/")
role_arn, image_uri = role_arn.strip(), image_uri.strip()
assert bucket and "/" not in bucket and role_arn and image_uri, "Fill in bucket, role_arn and image_uri."
pipeline_role_arn = job_role_arn = role_arn  # Set job_role_arn separately here if the platform provides one.
s3_root = f"s3://{bucket}/{namespace}"
print("Role for pipelines and jobs:", role_arn)
print("Will use S3 folder:", s3_root)


### 3. Deployment and platform settings

- Direct deploy uploads this working copy; reuse its name to replace source between executions.
- CI deploy requires a clean checkout and the reviewed commit supplied by CI; it uses a release/commit prefix.
- Only set VPC/KMS values when the platform supplies them. ResolveModel needs access to the SageMaker API.

In [ ]:
deploy_mode = "direct-deploy"
deploy_name = "my-review"           # Development slug, or CI release name
reviewed_commit = ""               # CI supplies this full hash; not a branch name
infrastructure = {
    "network": {
        "isolation": False,
        "subnet_ids": [],
        "security_group_ids": [],
    },
    "output_kms_key_arn": "",
    "volume_kms_key_arn": "",
}
identity = source_identity(deploy_mode, deploy_name, reviewed_commit)
source_uri = source_location(s3_root, identity)
input_root = s3_root.rstrip("/") + "/inputs"
print(identity)
print("Do not overwrite direct-deploy source while an execution is running.")


### 4a. Check CreatePipeline with the supplied role

- Create one uniquely named test pipeline with only a true condition.
- Pass the exact role_arn from the first cell; there is no reconstructed role ARN.
- This does not start an execution, launch a job or access S3.


In [ ]:
import uuid
access_name = namespace + "-access-" + uuid.uuid4().hex[:8]
access_definition = {
    "Version": "2020-12-01",
    "Steps": [{
        "Name": "CheckAccess", "Type": "Condition",
        "Arguments": {
            "Conditions": [{"Type": "Equals", "LeftValue": 1, "RightValue": 1}],
            "IfSteps": [], "ElseSteps": [],
        },
    }],
}
print("CreatePipeline:", access_name, "| supplied role:", role_arn)
access_pipeline = sm.create_pipeline(
    PipelineName=access_name, RoleArn=role_arn,
    PipelineDefinition=json.dumps(access_definition),
    Tags=[{"Key": "Diagnostic", "Value": "pipeline-publication"}],
)
print(access_pipeline)


### 4b. Check DescribePipeline and its saved role

- Read the test pipeline directly from SageMaker.
- Print the actual saved role; read the initial version, 1.
- Verify that the saved role is exactly the supplied ARN.


In [ ]:
access_description = sm.describe_pipeline(PipelineName=access_name, PipelineVersionId=1)
print("DescribePipeline:", access_description["PipelineArn"])
print("Saved role:", access_description["RoleArn"])
print("Read initial version: 1")
assert access_description["RoleArn"] == role_arn


### 4c. Check ordinary UpdatePipeline

- Update only the test pipeline, using the same supplied role and condition.
- Change its description to make this a visible update.
- Print the returned numeric version; no version-label call is involved.


In [ ]:
access_pipeline = sm.update_pipeline(
    PipelineName=access_name, RoleArn=role_arn,
    PipelineDefinition=json.dumps(access_definition),
    PipelineDescription="Ordinary UpdatePipeline access confirmed",
)
print("UpdatePipeline:", access_pipeline)


### 4d. Test the separate version-label permission

- Call UpdatePipelineVersion on the test pipeline's returned version.
- Print the full original AWS error if denied, then continue to publication.
- This only tests version labeling; the training/inference publisher below does not use it.


In [ ]:
try:
    print(sm.update_pipeline_version(
        PipelineArn=access_pipeline["PipelineArn"],
        PipelineVersionId=access_pipeline["PipelineVersionId"],
        PipelineVersionDisplayName="access-check",
        PipelineVersionDescription="Optional version-label permission check",
    ))
except sm.exceptions.ClientError as error:
    print(error)
    print("Version labeling failed. Continue: publication below does not call this API.")


### 5. Check the chosen S3 prefix

- Write a tiny probe under the application prefix using notebook credentials.
- Read the same object back; errors remain visible. The small probe file stays and is overwritten next run.
- This does not establish the job role's S3 access; managed execution checks that separately.

In [ ]:
bucket, prefix = s3_root.removeprefix("s3://").rstrip("/").split("/", 1)
assert s3_root.startswith("s3://") and prefix, "Use a bucket plus a nonempty application prefix."
probe_key = prefix + "/checks/source-deploy.txt"
s3.put_object(Bucket=bucket, Key=probe_key, Body=b"source deployment check")
print(s3.get_object(Bucket=bucket, Key=probe_key)["Body"].read().decode())


### 6. Pin the existing image

- Resolve a tag to its ECR digest; for a repository address, use its newest tagged image. Keep a supplied digest URI.
- Use that image for both training and Processing steps.
- No Docker build, image push or package installation happens here.

HPO cannot override the image's startup command. This source route requires an image
that runs `python -m mlops_models.entrypoints train` and honors `PYTHONPATH`.
Ordinary Training and Processing can override that command; their success alone
does not confirm HPO support. Confirm the platform image before selecting UseHPO.


In [ ]:
if "@sha256:" not in image_uri:
    registry, repository = image_uri.split("/", 1)
    repository, _, tag = repository.partition(":")
    ecr = session.client("ecr")
    if tag:
        detail = ecr.describe_images(
            registryId=registry.split(".")[0], repositoryName=repository,
            imageIds=[{"imageTag": tag}],
        )["imageDetails"][0]
    else:
        # Repository address only: use its newest tagged image, as 00G section 6 does.
        images = []
        for page in ecr.get_paginator("describe_images").paginate(
                registryId=registry.split(".")[0], repositoryName=repository,
                filter={"tagStatus": "TAGGED"}):
            images.extend(page["imageDetails"])
        assert images, "That repository has no tagged images; paste a full image URI instead."
        detail = max(images, key=lambda item: item["imagePushedAt"])
        print("Pinned the newest tagged image:", detail.get("imageTags"))
    image_uri = registry + "/" + repository + "@" + detail["imageDigest"]
print("Image:", image_uri)


### 7. Upload source and sample inputs

- Upload the whole mlops_models directory, including model folders and configuration.
- Copy the tracked runtime/load_source.py beside it; Processing and ordinary Training run this file first.
- Put synthetic inputs under inputs/<model>/; replace this input root with your own prepared CSV folders later.

The paths were defined in section 3. If this upload fails, you can still render
and attempt to publish the definitions in sections 8–9. This checks publication
access only; do not treat it as a runnable pipeline until uploads and job S3 access work.


In [ ]:
upload_source(session, s3_root, identity)
upload_examples(session, s3_root, model_ids)
print("Source:", source_uri)
print("Inputs:", input_root)


### 8. Build both definitions locally

- Reuse the common training/inference definitions and add the source inputs.
- ModelId selects a use case; UseHPO defaults to false and selects native tuning when true.
- InputDataRoot contains one subfolder per model.
- Training registers PendingManualApproval; inference resolves the highest Completed, Approved version.

In [ ]:
definitions = build_definitions(
    namespace, job_role_arn, image_uri, s3_root, source_uri, identity,
    region=session.region_name, models=model_ids, infrastructure=infrastructure, batch_runtime=batch_runtime,
)
WORK = ROOT / ".runs/source-deploy" / namespace
WORK.mkdir(parents=True, exist_ok=True)
for kind, body in definitions.items():
    path = WORK / (kind + ".json")
    path.write_text(json.dumps(body, indent=2) + "\n")
    print(namespace + "-" + kind, "→", " → ".join(step["Name"] for step in body["Steps"]))
    print("Definition:", path.resolve())
print("Registry groups:", [namespace + "-" + model for model in model_ids])


### 9. Publish both pipelines without version-label calls

- Create or update only pipelines tagged for this namespace and deployment mode.
- A new pipeline starts at version 1; UpdatePipeline returns the updated version ID.
- Read back that exact version and verify its definition and saved role.
- Record both published pipelines for Studio and optional inspection; no version-list or version-label call is made.


In [ ]:
published = {}
for kind, body in definitions.items():
    name = f"{namespace}-{kind}"
    request = {
        "PipelineName": name, "RoleArn": pipeline_role_arn,
        "PipelineDefinition": json.dumps(body),
    }
    try:
        existing = sm.describe_pipeline(PipelineName=name)
    except sm.exceptions.ClientError as error:
        if "does not exist" not in error.response["Error"]["Message"]:
            raise
        response = sm.create_pipeline(**request, Tags=[
            {"Key": "Deployment", "Value": namespace},
            {"Key": "SourceDeploy", "Value": identity["mode"]},
        ])
        version = 1  # The initial version of a newly created pipeline.
    else:
        tags = {t["Key"]: t["Value"] for t in sm.list_tags(ResourceArn=existing["PipelineArn"])["Tags"]}
        if tags.get("Deployment") != namespace or tags.get("SourceDeploy") != identity["mode"]:
            raise ValueError(f"{name} belongs to another deployment.")
        response = sm.update_pipeline(**request)
        version = response["PipelineVersionId"]
    actual = sm.describe_pipeline(PipelineName=name, PipelineVersionId=version)
    assert json.loads(actual["PipelineDefinition"]) == body, "Published definition differs; inspect before executing."
    assert actual["RoleArn"] == pipeline_role_arn, "Saved pipeline role differs; inspect before executing."
    published[kind] = {"name": name, "arn": response["PipelineArn"], "version": version}
    print("Published:", name, "| version:", version, "| saved role:", actual["RoleArn"])

publication = {
    "namespace": namespace, "pipelines": published, "models": model_ids,
    "source": identity, "source_uri": source_uri, "image": image_uri,
    "s3_root": s3_root, "batch_runtime": batch_runtime, "region": session.region_name,
}
path = WORK / "publication.json"
path.write_text(json.dumps(publication, indent=2) + "\n")
print("Publication complete:", path.resolve())
print("Both pipelines are ready to start from Studio.")


### 10. Remove only the test pipeline

- Delete the uniquely named access-check pipeline created in section 4a.
- Keep the published training and inference pipelines.
- A cleanup denial does not undo publication; the printed test name identifies what remains.


In [ ]:
print("DeletePipeline:", access_name)
print(sm.delete_pipeline(PipelineName=access_name))


### Continue in Studio

1. Open **Pipelines → `<namespace>-training` → Create execution**. Choose `ModelId`, and name the execution `<ModelId>-training-<label>`.
   For tuning, set `UseHPO=true`, choose the trial limits, and use `<ModelId>-hpo-<label>`.
2. Inspect the evaluation report and the registered model under **Model Registry → `<namespace>-<model>`**.
3. Approve the version after reviewing it. Training finishes while approval is pending.
4. Open **`<namespace>-inference` → Create execution** and select the same model. Name it `<ModelId>-inference-<label>`.
   Inference records its exact approved package under the run's `reports/authorization`.
   It fails before scoring if none is approved.

`InputDataRoot` expects `<root>/<ModelId>/` CSV inputs.
Outputs use `<OutputDataRoot>/<ModelId>/runs/<execution-id>/`.
The optional 03G notebook can start and inspect these same executions.


The shared HPO search ranges come from `model-config.json` and must match across
the selected models. The objective metric and direction come from the selected
model at execution, so regression and classification can use different metrics.
Changing the search space requires republishing; changing UseHPO does not.

### Optional: inspect the image startup configuration

- Reuse the exact image URI printed above; read its manifest and configuration through ECR.
- For an image with multiple architectures, select Linux amd64 for the default CPU instance.
- Print only the image user, entrypoint, default arguments and working directory. This reads metadata without starting a job or changing permissions.
- A script in Entrypoint or Cmd must be inspected separately to find out whether it prepares output-directory permissions.


In [ ]:
import boto3
import json
from urllib.request import urlopen

# Use the resolved, digest-pinned image from this notebook.
assert "@sha256:" in image_uri, "Run the image-resolution cell first so image_uri includes @sha256:."
image_registry, image_reference = image_uri.split("/", 1)
image_repository, image_digest = image_reference.split("@", 1)
image_ecr = boto3.client("ecr", region_name=image_registry.split(".")[3])

image_response = image_ecr.batch_get_image(
    registryId=image_registry.split(".")[0],
    repositoryName=image_repository,
    imageIds=[{"imageDigest": image_digest}],
)
assert image_response.get("images"), image_response.get("failures")
image_manifest = json.loads(image_response["images"][0]["imageManifest"])

# An image index contains a separate manifest for each architecture.
if "manifests" in image_manifest:
    image_platforms = [
        item for item in image_manifest["manifests"]
        if item.get("platform", {}).get("os") == "linux"
        and item.get("platform", {}).get("architecture") == "amd64"
    ]
    assert len(image_platforms) == 1, "Expected one Linux amd64 image; inspect the image manifest."
    image_response = image_ecr.batch_get_image(
        registryId=image_registry.split(".")[0],
        repositoryName=image_repository,
        imageIds=[{"imageDigest": image_platforms[0]["digest"]}],
    )
    assert image_response.get("images"), image_response.get("failures")
    image_manifest = json.loads(image_response["images"][0]["imageManifest"])

image_download = image_ecr.get_download_url_for_layer(
    registryId=image_registry.split(".")[0],
    repositoryName=image_repository,
    layerDigest=image_manifest["config"]["digest"],
)
with urlopen(image_download["downloadUrl"], timeout=20) as stream:
    image_config = json.load(stream)["config"]

print("Image:", image_uri)
for field in ("User", "Entrypoint", "Cmd", "WorkingDir"):
    print(f"{field}: {image_config.get(field)}")


### Optional: read runner.py from the image

- Run the startup-configuration cell above first; reuse its ECR client, image manifest and working directory.
- Read the newest image layers first and locate runner.py. Layer downloads may take a few minutes for a large image.
- Print the script with line numbers here for review. The script is not executed, no files are extracted, and no jobs or permissions are changed.
- Stop if the image deletes the script or uses a link instead; do not display an older copy as the current startup script. Keep customer source and saved notebook outputs inside the customer environment.


In [ ]:
import tarfile
from pathlib import PurePosixPath
from urllib.request import urlopen

assert "runner.py" in (image_config.get("Entrypoint") or []), "Check Entrypoint above: this cell reads runner.py."
runner_path = PurePosixPath(image_config.get("WorkingDir") or "/") / "runner.py"
runner_name = runner_path.as_posix().lstrip("/")
print("Reading from the image:", runner_path, flush=True)

# OCI deletion markers hide files from older layers, including deleted parent folders.
runner_parents = {parent.as_posix().lstrip("/") for parent in runner_path.parents}
runner_deleted_markers = set()
for path in (runner_path, *runner_path.parents):
    if path.name:
        runner_deleted_markers.add((path.parent / (".wh." + path.name)).as_posix().lstrip("/"))
for parent in runner_path.parents:
    runner_deleted_markers.add((parent / ".wh..wh..opq").as_posix().lstrip("/"))

runner_text = None
for layer_index in range(len(image_manifest["layers"]) - 1, -1, -1):
    layer = image_manifest["layers"][layer_index]
    print(f"Checking image layer {layer_index + 1}/{len(image_manifest['layers'])}", flush=True)
    layer_download = image_ecr.get_download_url_for_layer(
        registryId=image_registry.split(".")[0],
        repositoryName=image_repository,
        layerDigest=layer["digest"],
    )
    runner_hidden = False
    with urlopen(layer_download["downloadUrl"], timeout=30) as stream:
        with tarfile.open(fileobj=stream, mode="r|*") as archive:
            for member in archive:
                member_name = PurePosixPath(member.name).as_posix().lstrip("/")
                if member_name in runner_deleted_markers:
                    runner_hidden = True
                if member_name in runner_parents and not member.isdir():
                    raise ValueError(f"Startup path uses a link or non-directory: {member.name}; inspect with the image owner.")
                if member_name == runner_name:
                    if not member.isfile():
                        raise ValueError(f"{runner_path} is not a regular file; inspect its link target with the image owner.")
                    if member.size > 128 * 1024:
                        raise ValueError("runner.py exceeds the 128 KiB display limit; review it in the image-build repository.")
                    with archive.extractfile(member) as script:
                        runner_text = script.read().decode("utf-8")
    if runner_text is not None:
        break
    if runner_hidden:
        raise FileNotFoundError(f"{runner_path} is hidden by an image-layer deletion; refusing to show an older copy.")

if runner_text is None:
    raise FileNotFoundError(f"{runner_path} was not found in the image layers.")
print(f"\n--- {runner_path} ---")
for line_number, line in enumerate(runner_text.splitlines(), 1):
    print(f"{line_number:4}: {line}")
